In [ ]:
!pip install ultralytics flask flask-ngrok pyngrok

In [ ]:
from flask import Flask, request, jsonify
from pyngrok import ngrok
from ultralytics import YOLO
import cv2
import numpy as np
import base64
from collections import defaultdict
import time

# NGROK SERVER(REQUIRED)
ngrok.set_auth_token("3ASqrcPQ7WGZeYryJZnuRu7zcoK_7vgmtG5DHyZfPHArEc8DX")

# FLASK INITIALIZE(REQUIRED)
app = Flask(__name__)

public_url = ngrok.connect(5000).public_url
print("=========================================================")
print(f"COPY THIS EXACT URL TO YOUR LAPTOP SCRIPT:")
print(f"{public_url}/detect")
print("=========================================================")

# PATH NG MODEL(REQUIRED)
MODEL_PATH = '/content/drive/MyDrive/Colab_Notebooks/my_model.pt'

try:
    print("Loading YOLO model...")
    model = YOLO(MODEL_PATH)
    print("Model Loaded!")
except Exception as e:
    print(f"Error loading model: {e}. Downloading YOLOv8n fallback.")
    model = YOLO('my_model.pt')

@app.route('/detect', methods=['POST'])
def detect():
    start_time = time.time()

    data = request.json
    if not data or 'frame' not in data:
        return jsonify({'error': 'No frame provided'}), 400

    try:
        # Decode the base64 string back into an image
        img_data = base64.b64decode(data['frame'])
        np_arr = np.frombuffer(img_data, np.uint8)
        frame = cv2.imdecode(np_arr, cv2.IMREAD_COLOR)

        # Run YOLO Inference
        results = model(frame, conf=0.5, verbose=False)

        # Extract Data
        detection_info = {'total': 0, 'labels': defaultdict(int)}
        for r in results:
            for box in r.boxes:
                c = int(box.cls)
                label = model.names[c]
                detection_info['labels'][label] += 1
                detection_info['total'] += 1

        detection_info['labels'] = dict(detection_info['labels'])

        # Calculate Speed
        detection_time = time.time() - start_time
        detection_info['inference_ms'] = round(detection_time * 1000)

        return jsonify(detection_info)

    except Exception as e:
        return jsonify({'error': str(e)}), 500

if __name__ == '__main__':
    print("Starting Flask Server Colab")
    app.run(port=5000)